# Wayfinder: model runs on a Colab GPU

A thin wrapper. Every cell calls a script in `src/`, so the notebook holds no logic of its own and the runs are
reproducible from the command line on any GPU machine.

It runs the three model methods on the frozen test split:
1. **Zero-shot**: `src/run_baseline.py zero_shot`
2. **Few-shot**: `src/run_baseline.py few_shot` (needs the hand-checked `data/gold/fewshot_examples.json`)
3. **QLoRA fine-tune**: `src/train_lora.py train`, then `predict`
4. **Distilled fine-tune**: `src/distill.py label`/`build`, then `train_lora.py` on the distilled labels

**Before you start**
- Runtime → Change runtime type → **T4 GPU**.
- Add a read-only GitHub token with access to the private `rafgaf-dev/wayfinder-data` repo as the Colab
  secret `GITHUB_TOKEN`. The texts the models read live there; the public repo holds labels only.
- Predictions and the adapter are written to Google Drive, so if the session dies, re-running the same cell resumes.

Rough times on a T4: zero-shot ~15 min, few-shot ~20 min, training ~1-1.5 h, fine-tune predictions ~10 min.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Setup

In [ ]:
REPO_URL = "https://github.com/rafgaf-dev/wayfinder.git"  # public: code, labels, results
# The texts the models read are in the private rafgaf-dev/wayfinder-data repo, because they are third-party
# text (store copy, event listings) whose terms do not allow republishing. Add a read-only GitHub token with
# access to it as a Colab secret named GITHUB_TOKEN (key icon in the left sidebar).
import os
from google.colab import userdata
token = userdata.get("GITHUB_TOKEN")
if not os.path.exists("/content/wayfinder"):
    !git clone --depth 1 {REPO_URL} /content/wayfinder
%cd /content/wayfinder
if not os.path.exists("data/private"):
    !git clone --depth 1 https://{token}@github.com/rafgaf-dev/wayfinder-data.git data/private
!git log -1 --oneline && git -C data/private log -1 --oneline
!ls data/private/processed


In [ ]:
!pip install -q -r requirements.txt -r requirements-gpu.txt
# Colab preinstalls an old torchao, which this pipeline does not use; peft raises an ImportError on seeing it
# when attaching an adapter to the fp16 model for serving. Removing it makes peft skip that path.
!pip uninstall -y -q torchao
!python -c "import torch, transformers, peft, bitsandbytes; print(torch.__version__, transformers.__version__, peft.__version__, bitsandbytes.__version__)"

In [ ]:
# Keep everything that must survive a session on Drive. results/predictions in the clone becomes a link to
# Drive, so the scripts write straight there and resume from there.
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/wayfinder"
!mkdir -p {DRIVE}/predictions {DRIVE}/lora
!rm -rf results/predictions && mkdir -p results && ln -s {DRIVE}/predictions results/predictions
!ls -la results/

## Smoke test

Five items, to catch problems in a minute rather than an hour. It writes to its own run name, so it never mixes
with the real run.

In [ ]:
!python src/run_baseline.py zero_shot --limit 5
!head -c 600 results/predictions/zero_shot-limit5.jsonl

## Baselines

In [ ]:
!python src/run_baseline.py zero_shot

In [ ]:
!python src/run_baseline.py few_shot

## Fine-tune

`train` saves checkpoints to Drive every 100 steps; `--resume` continues from the latest one if the session died,
and starts fresh if there is none. `predict` merges the adapter into the fp16 base, so the fine-tune is served
exactly like the baselines.

In [ ]:
!python src/train_lora.py train --output-dir {DRIVE}/lora --resume

In [ ]:
!python src/train_lora.py predict --output-dir {DRIVE}/lora
!cp {DRIVE}/lora/training_summary.json {DRIVE}/predictions/lora.training_summary.json

### Optional: serving on the 4-bit base

Measures the QLoRA caveat instead of assuming it: the adapter was trained against the quantised weights, and the
run above applies it to the fp16 ones.

In [ ]:
!python src/train_lora.py predict --output-dir {DRIVE}/lora --serve quantised

## Distilled fine-tune

A second fine-tune whose mood and commitment labels come from the few-shot model (the teacher) instead of the
silver rules; every other axis stays silver. It needs the hand-checked few-shot examples, so run it after the
few-shot baseline. The teacher's outputs are written to Drive as they arrive, so a dead session resumes.
Roughly 1.5 h for train, 2 min for validation, then training and predictions as for the first fine-tune.

In [ ]:
!python src/distill.py label train

In [ ]:
!python src/distill.py label val
!python src/distill.py build
!cp results/distill_report.json {DRIVE}/predictions/

In [ ]:
!python src/train_lora.py train --labels data/private/processed/train_distilled.jsonl \
    --val-labels data/private/processed/val_distilled.jsonl --output-dir {DRIVE}/lora-distilled --resume

In [ ]:
!python src/train_lora.py predict --output-dir {DRIVE}/lora-distilled --run-name lora-distilled
!cp {DRIVE}/lora-distilled/training_summary.json {DRIVE}/predictions/lora-distilled.training_summary.json

## Getting the results back

Everything is in `MyDrive/wayfinder/predictions/` on Drive. Download those files into the repo's
`results/predictions/` locally, then score each run there, where the finished gold labels are:

```
.venv/bin/python src/evaluate.py score results/predictions/zero_shot.jsonl
.venv/bin/python src/evaluate.py compare results/predictions/few_shot.jsonl results/predictions/lora.jsonl
```